In [1]:
from vv.data import CIFAR10Loader, MNISTLoader
from vv.models import Autoencoder
from vv.models.fundamental import *
from vv.utilities import get_dir
import torch.nn.functional as F
import torch
from torchviz import make_dot

/home/computerman/Desktop/VQ-VAE/VQ-VAE-Praktikum-SS24-HD/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = "CIFAR10"

# Data

In [3]:
if data == "CIFAR10":
    data_loader = CIFAR10Loader(train_val_split=0.8,
                                batch_size=1,
                                workers=10,
                                verbose=False)
    data_size = [3, 32, 32]
elif data == "MNIST":
    data_loader = MNISTLoader(train_val_split=0.8,
                            batch_size=1,
                            workers=10,
                            verbose=False)
    data_size = [1, 28, 28]

In [4]:
data_loader.prepare_data()
data_loader.setup("predict")
data = data_loader.predict_dataloader()


sample = next(iter(data))
x = sample[0]
print(sample[0].shape)
print(sample[1].shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([1, 3, 32, 32])
torch.Size([1])


# Model

### Base encoder

In [5]:
base_encoder = ResEncoder(input_size=data_size,
                          output_units=10,
                          dropout_prob=0.0,
                          batch_norm=True)

### Encoder

In [ ]:
encoder = base_encoder

In [6]:
encoder = VQEncoder(base_encoder = base_encoder,
                    codebook_splits=[10, 1, 1],
                    codebook_size=512,
                    decoder_encoder_ste=True,
                    use_commitment_loss=True,
                    vq_loss="mse",
                    embedding_loss_weight=0.25)

### Decoder

In [7]:
decoder = ResDecoder(output_size=data_size,
                     input_units=10,
                     dropout_prob=0.0,
                     batch_norm=True)

### Full model

In [8]:
model = Autoencoder(encoder=encoder,
                    decoder=decoder,
                    disconnect_decoder_loss_propagation_from_encoder=False,
                    latent_error_weight=1.0,
                    learning_rate=1e-3,
                    lr_patience=5,
                    check_val_every_n_epoch=5)

# Gradient Flow

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
x = x.to(device)

In [10]:
model.visualize_gradient_flow(x=x,
                              filepath="tmp/gradient_flow_grad",
                              show_gradients=True)
model.visualize_gradient_flow(x=x,
                              filepath="tmp/gradient_flow",
                              show_gradients=False)
